#  LSH - Tìm Ví Tương Đồng từ ALS Model

**Pipeline:** ALS userFactors → LSH Bucketing → Tìm ví tương đồng



## Cài đặt môi trường

In [1]:
# Cài PySpark
!pip install pyspark==3.5.0 -q
print(' PySpark installed')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.9/316.9 MB 4.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 22.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.0 which is incompatible.
 PySpark installed


## Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

#  Sửa đường dẫn này cho đúng với vị trí thư mục als_model trong Drive của bạn
ALS_MODEL_PATH = '/content/drive/MyDrive/als_model (1)'

import os
assert os.path.exists(ALS_MODEL_PATH), f' Không tìm thấy: {ALS_MODEL_PATH}'
print(f' Found model at: {ALS_MODEL_PATH}')
print('Contents:', os.listdir(ALS_MODEL_PATH))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
 Found model at: /content/drive/MyDrive/als_model (1)
Contents: ['metadata', 'userFactors', 'itemFactors']


## Khởi tạo SparkSession

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName('LSH_WalletSimilarity') \
    .config('spark.driver.memory', '8g') \
    .config('spark.executor.memory', '8g') \
    .config('spark.sql.adaptive.enabled', 'true') \
    .config('spark.sql.adaptive.coalescePartitions.enabled', 'true') \
    .config('spark.sql.autoBroadcastJoinThreshold', '100mb') \
    .getOrCreate()

spark.sparkContext.setLogLevel('WARN')
print(f' Spark {spark.version} ready')

 Spark 3.5.0 ready


## Đọc userFactors từ ALS Model

In [3]:
user_factors_path = f'{ALS_MODEL_PATH}/userFactors'

df_users = spark.read.parquet(user_factors_path)

print('Schema:')
df_users.printSchema()
print(f'\nTổng số ví: {df_users.count():,}')
df_users.show(5, truncate=True)

Schema:
root
 |-- id: integer (nullable = true)
 |-- features: array (nullable = true)
 |    |-- element: float (containsNull = true)


Tổng số ví: 31,366
+---+--------------------+
| id|            features|
+---+--------------------+
|  6|[0.04042604, 0.24...|
| 16|[0.56151336, 0.0,...|
| 26|[0.0, 0.0, 0.0, 0...|
| 36|[0.11013239, 0.0,...|
| 46|[0.0, 0.0, 0.0, 0...|
+---+--------------------+
only showing top 5 rows



## Chuyển Array → DenseVector

In [4]:
from pyspark.ml.linalg import Vectors, VectorUDT
from pyspark.sql.functions import udf, col

# UDF chuyển list/array sang DenseVector
list_to_vector_udf = udf(lambda l: Vectors.dense(l), VectorUDT())

df_users_vec = df_users.withColumn('features_vec', list_to_vector_udf(col('features')))

# Cache để tái sử dụng
df_users_vec.cache()
df_users_vec.count()  # trigger cache

print(' Chuyển đổi hoàn tất')
df_users_vec.select('id', 'features_vec').show(3, truncate=80)

 Chuyển đổi hoàn tất
+---+--------------------------------------------------------------------------------+
| id|                                                                    features_vec|
+---+--------------------------------------------------------------------------------+
|  6|[0.04042603820562363,0.24310679733753204,0.02734369970858097,0.0,0.0,0.098762...|
| 16|[0.561513364315033,0.0,0.019771473482251167,0.0,0.0,0.20332083106040955,0.0,0...|
| 26|[0.0,0.0,0.0,0.0,0.052685998380184174,0.21825754642486572,0.0,0.0,0.0,0.09111...|
+---+--------------------------------------------------------------------------------+
only showing top 3 rows



## Huấn luyện mô hình LSH

In [5]:
from pyspark.ml.feature import BucketedRandomProjectionLSH

#  Tuning parameters:
# - bucketLength: Tăng → thùng rộng hơn, nhiều ứng viên hơn, recall cao hơn nhưng precision giảm
# - numHashTables: Tăng → chính xác hơn nhưng tốn RAM/thời gian hơn
BUCKET_LENGTH  = 0.5
NUM_HASH_TABLES = 3

brp = BucketedRandomProjectionLSH(
    inputCol='features_vec',
    outputCol='hashes',
    bucketLength=BUCKET_LENGTH,
    numHashTables=NUM_HASH_TABLES,
    seed=42
)

print(' Đang fit LSH model...')
lsh_model = brp.fit(df_users_vec)
print(' LSH model ready')

# Xem kết quả băm thử
df_hashed = lsh_model.transform(df_users_vec)
df_hashed.select('id', 'hashes').show(5, truncate=80)

 Đang fit LSH model...
 LSH model ready
+---+------------------------+
| id|                  hashes|
+---+------------------------+
|  6| [[-1.0], [-1.0], [0.0]]|
| 16|   [[0.0], [0.0], [0.0]]|
| 26|[[-1.0], [-1.0], [-1.0]]|
| 36|  [[-2.0], [0.0], [1.0]]|
| 46|  [[-1.0], [0.0], [1.0]]|
+---+------------------------+
only showing top 5 rows



##  Tìm ví tương đồng cho một ví cụ thể

In [6]:

TARGET_USER_ID   = None
TOP_N_NEIGHBORS  = 10

# lấy id đầu tiên trong dataset
if TARGET_USER_ID is None:
    TARGET_USER_ID = df_users_vec.first()['id']
    print(f'  Tự động chọn TARGET_USER_ID = {TARGET_USER_ID}')

# Lấy vector của ví mục tiêu
target_row = df_users_vec.filter(col('id') == TARGET_USER_ID).first()

if target_row is None:
    raise ValueError(f' Không tìm thấy user_id = {TARGET_USER_ID}')

target_vector = target_row['features_vec']
print(f' Tìm thấy ví {TARGET_USER_ID} | Vector dim: {len(target_vector)}')

  Tự động chọn TARGET_USER_ID = 6
 Tìm thấy ví 6 | Vector dim: 50


In [7]:
# Chạy LSH Approximate Nearest Neighbors
print(f' Tìm {TOP_N_NEIGHBORS} ví tương đồng nhất với ví {TARGET_USER_ID}...')

similar_users = lsh_model.approxNearestNeighbors(
    dataset=df_users_vec,
    key=target_vector,
    numNearestNeighbors=TOP_N_NEIGHBORS + 1,  # +1 vì kết quả sẽ chứa chính ví đó
    distCol='euclidean_dist'
)

# Loại bỏ chính ví mục tiêu (khoảng cách = 0)
result = similar_users.filter(col('id') != TARGET_USER_ID) \
                       .select('id', 'euclidean_dist') \
                       .orderBy('euclidean_dist')

print(f'\n Top {TOP_N_NEIGHBORS} ví tương đồng với ví {TARGET_USER_ID}:')
result.show(TOP_N_NEIGHBORS, truncate=False)

 Tìm 10 ví tương đồng nhất với ví 6...

 Top 10 ví tương đồng với ví 6:
+---+------------------+
|id |euclidean_dist    |
+---+------------------+
|4  |0.5752922934554199|
|1  |0.7029152444748268|
|7  |0.8247287741726853|
|2  |0.8492768077317546|
|0  |0.8579631818760365|
|3  |0.8737272058735976|
|20 |1.0330002423187494|
|14 |1.0626326426413966|
|28 |1.0726713038198816|
|10 |1.089288472557539 |
+---+------------------+



##  Similarity Join - Tìm tất cả cặp ví tương đồng trong toàn bộ dataset

In [9]:
from pyspark.sql.functions import col

TEST_THRESHOLD = 0.1

print(f'Đang tìm cặp ví (Threshold = {TEST_THRESHOLD})...')

# 2. Thực hiện LSH Join
all_similar_pairs = lsh_model.approxSimilarityJoin(
    df_users_vec,
    df_users_vec,
    threshold=TEST_THRESHOLD,
    distCol='dist'
)

# 3. Lọc bỏ cặp trùng lặp (Ví A - Ví A)
filtered_pairs = all_similar_pairs.filter(col('datasetA.id') < col('datasetB.id')) \
    .select(
        col('datasetA.id').alias('user_a'),
        col('datasetB.id').alias('user_b'),
        col('dist')
    )

# 4. Cache lại vào RAM của Colab
filtered_pairs.cache()

# 5. Đếm xem ra bao nhiêu cặp
total_pairs = filtered_pairs.count()
print(f'\nTổng số cặp ví tương đồng: {total_pairs:,}')

# 6. Chỉ hiển thị 20 dòng đầu tiên (không cần sort toàn bộ 450 triệu dòng)
if total_pairs > 0:
    filtered_pairs.show(20, truncate=False)
else:
    print("Không tìm thấy cặp nào. Hãy tăng nhẹ TEST_THRESHOLD lên (ví dụ: 0.8 hoặc 1.0)")

Đang tìm cặp ví (Threshold = 0.1)...

Tổng số cặp ví tương đồng: 22,405,256
+------+------+---------------------+
|user_a|user_b|dist                 |
+------+------+---------------------+
|938   |7925  |0.0011633114646785474|
|938   |12976 |0.0011633114646785474|
|938   |13905 |0.0011633114646785474|
|938   |15767 |0.0011633114646785474|
|1013  |18615 |0.0043950895759912855|
|1207  |6125  |7.239747146282829E-4 |
|1207  |16782 |7.239747146282829E-4 |
|1207  |19082 |7.239747146282829E-4 |
|1207  |23315 |7.239747146282829E-4 |
|1207  |26296 |7.239747146282829E-4 |
|1207  |27656 |7.239747146282829E-4 |
|1207  |30565 |7.239747146282829E-4 |
|1210  |16967 |0.005136324507465937 |
|1210  |21292 |0.005136324507465937 |
|1226  |3152  |4.164550629519165E-4 |
|1226  |9795  |8.862540796472543E-4 |
|1226  |10267 |8.859865584233888E-4 |
|1226  |13856 |8.862540796472543E-4 |
|1226  |15532 |8.862540796472543E-4 |
|1226  |16592 |8.862540796472543E-4 |
+------+------+---------------------+
only showing

## Lưu kết quả